# 01 — Animal Overview Reports

Per-animal PDF reports with session timeline, trajectories, psychometric curves (pairwise),
update matrices, Hard-A vs Hard-B comparisons, and strip-plot summary.

**Structure:**
1. `compute_phase` — data for one animal x one phase
2. `plot_*` — individual figure per page type
3. `generate_animal_report` — assembles all pages for one animal
4. Batch loop — saves one PDF per animal

## Setup

In [ ]:
%matplotlib inline
from shared_setup import *
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.lines import Line2D

from analysis.phase import (compute_phase, downsample_phase, build_strip_df,
                            is_opto_cohort, filter_phase, PHASE_ORDER)
from behav_utils.analysis.downsample import calculate_min_n
from plotting._style import CONDITION_COLOURS as CC, DIST_COLOURS, TYPE_MARKERS

experiment, info = load_data()
print(f"Mode: {info['mode']}")

## Cohort Summary

In [ ]:
rows = []
for aid, animal in sorted(experiment.animals.items()):
    table = animal.session_table
    types = dict(table.session_type.value_counts())
    dists = dict(table.distribution.value_counts())
    rows.append({
        'animal_id': aid, 'genotype': animal.genotype,
        'n_sessions': len(table),
        'n_uniform': dists.get('Uniform', 0),
        'n_hard_a': dists.get('Hard-A', 0),
        'n_hard_b': dists.get('Hard-B', 0),
        'regular': types.get('regular', 0),
        'masking': types.get('masking', 0),
        'opto': types.get('opto', 0),
        'washout': types.get('washout', 0),
    })
cohort_df = pd.DataFrame(rows)
print(cohort_df.to_string(index=False))

## Colours

Colours come from the style modules, not inline hex: condition colours from `plotting/_style.py` (`CONDITION_COLOURS`, imported as `CC` in Setup), distribution colours and session-type markers also from `plotting/_style.py` (`DIST_COLOURS`, `TYPE_MARKERS`). Only the Hard-A / Hard-B convenience aliases are derived here.

In [ ]:
# Condition colours (CC) and DIST_COLOURS / TYPE_MARKERS are imported in Setup.
HARD_A_COLOUR = DIST_COLOURS['Hard-A']
HARD_B_COLOUR = DIST_COLOURS['Hard-B']

## Plot Functions

In [ ]:
def _plot_pair(ax, psyc_dict, key_a, key_b, title,
               colour_a=None, colour_b=None, label_a=None, label_b=None):
    pa, pb = psyc_dict.get(key_a), psyc_dict.get(key_b)
    ca = colour_a or CC.get(key_a, 'grey')
    cb = colour_b or CC.get(key_b, 'grey')
    la = label_a or key_a
    lb = label_b or key_b
    if pa and pa.get('success'):
        plot_psychometric(pa, ax=ax, color=ca, label=f'{la} (n={pa["n_trials"]})')
    if pb and pb.get('success'):
        plot_psychometric(pb, ax=ax, color=cb, label=f'{lb} (n={pb["n_trials"]})')
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=7)

In [ ]:
def plot_session_timeline(animal, aid):
    table = animal.session_table
    fig, ax = plt.subplots(figsize=(12, 3))
    for _, row in table.iterrows():
        ax.scatter(row['session_idx'], row['accuracy'],
                   c=DIST_COLOURS.get(row['distribution'], 'grey'),
                   marker=TYPE_MARKERS.get(row['session_type'], 'o'),
                   s=60, edgecolors='k', linewidths=0.5, zorder=3)
    ax.set_xlabel('Session index'); ax.set_ylabel('Accuracy')
    ax.set_ylim(0.4, 1.0); ax.axhline(0.5, ls='--', c='grey', lw=0.5)
    handles = ([Line2D([0],[0], marker='o', color='w', markerfacecolor=c, markersize=8, label=d)
                for d, c in DIST_COLOURS.items()] +
               [Line2D([0],[0], marker=m, color='w', markerfacecolor='grey', markersize=8, label=t)
                for t, m in TYPE_MARKERS.items()])
    ax.legend(handles=handles, ncol=4, fontsize=8, loc='lower right')
    fig.suptitle(f'{aid} ({animal.genotype.upper()}) — Session Timeline', fontsize=14)
    fig.tight_layout()
    return fig

In [ ]:
# Session-trajectory page. 'mu' is the PSE — it comes from the per-session
# 'psychometric' fit, so 'psychometric' must be in the requested stats.
# Swap in any names from list_available_stats(); use 'mu' for the PSE.
# The two lists correspond position-for-position (request -> panel to draw).
TRAJECTORY_STATS  = ['accuracy', 'psychometric', 'win_stay', 'lose_shift', 'recency']
TRAJECTORY_PANELS = ['accuracy', 'mu',           'win_stay', 'lose_shift', 'recency']


def plot_trajectory_page(animal, tag):
    traj = compute_trajectory(animal, stat_names=TRAJECTORY_STATS)
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    for ax, stat in zip(axes.flat, TRAJECTORY_PANELS):
        plot_trajectory(traj, stat_name=stat, ax=ax, show_distribution_boundaries=True)
    for ax in axes.flat[len(TRAJECTORY_PANELS):]:
        ax.set_axis_off()
    fig.suptitle(f'{tag} — Session Trajectories', fontsize=14)
    fig.tight_layout()
    return fig

In [ ]:
def plot_uniform_psychometric(psyc, tag):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    _plot_pair(axes[0,0], psyc, 'baseline', 'masking',   'Baseline vs Masking')
    _plot_pair(axes[0,1], psyc, 'masking',  'all_opto',  'Masking vs All Opto')
    _plot_pair(axes[0,2], psyc, 'masking',  'opto_off',  'Masking vs Opto-Off')
    _plot_pair(axes[1,0], psyc, 'opto_off', 'opto_on',   'Opto-Off vs Opto-On')
    _plot_pair(axes[1,1], psyc, 'opto_on',  'post_opto', 'Opto-On vs Post-Opto')
    _plot_pair(axes[1,2], psyc, 'opto_off', 'post_opto', 'Opto-Off vs Post-Opto')
    fig.suptitle(f'{tag} — Uniform Psychometric', fontsize=14)
    fig.tight_layout()
    return fig

In [ ]:
def plot_uniform_um(um, tag):
    order = ['baseline', 'masking', 'all_opto', 'opto_off', 'opto_on', 'post_opto']
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    for i, key in enumerate(order):
        ax = axes[i // 3, i % 3]
        u = um.get(key)
        if u is not None:
            plot_um(u, ax=ax, title=f'{key} (n={u["n_trials"]})',
                    vmin=-0.35, vmax=0.35, colorbar=False)
        else:
            ax.text(0.5, 0.5, f'{key}\nNo data', ha='center', va='center',
                    transform=ax.transAxes, color='grey')
            ax.set_axis_off()
    fig.suptitle(f'{tag} — Uniform Update Matrices', fontsize=14)
    fig.tight_layout()
    return fig

In [ ]:
def plot_hard_psychometric(psyc_a, psyc_b, tag):
    fig, axes = plt.subplots(2, 4, figsize=(22, 10))
    _plot_pair(axes[0,0], psyc_a, 'masking', 'opto_off', 'Hard-A: Masking vs Opto-Off')
    _plot_pair(axes[0,1], psyc_b, 'masking', 'opto_off', 'Hard-B: Masking vs Opto-Off')
    for col, key, title in [(2, 'masking', 'A vs B: Masking'),
                             (3, 'opto_off', 'A vs B: Opto-Off')]:
        ax = axes[0, col]
        pa, pb = psyc_a.get(key), psyc_b.get(key)
        if pa and pa.get('success'):
            plot_psychometric(pa, ax=ax, color=HARD_A_COLOUR, label=f'Hard-A (n={pa["n_trials"]})')
        if pb and pb.get('success'):
            plot_psychometric(pb, ax=ax, color=HARD_B_COLOUR, label=f'Hard-B (n={pb["n_trials"]})')
        ax.set_title(title, fontsize=11); ax.legend(fontsize=7)
    _plot_pair(axes[1,0], psyc_a, 'opto_off', 'opto_on',   'Hard-A: Opto-Off vs Opto-On')
    _plot_pair(axes[1,1], psyc_b, 'opto_off', 'opto_on',   'Hard-B: Opto-Off vs Opto-On')
    _plot_pair(axes[1,2], psyc_a, 'opto_on',  'post_opto', 'Hard-A: Opto-On vs Post-Opto')
    _plot_pair(axes[1,3], psyc_b, 'opto_on',  'post_opto', 'Hard-B: Opto-On vs Post-Opto')
    fig.suptitle(f'{tag} — Hard Psychometric', fontsize=14)
    fig.tight_layout()
    return fig

In [ ]:
def plot_hard_um(um_a, um_b, tag):
    order = ['masking', 'all_opto', 'opto_off', 'opto_on', 'post_opto']
    fig, axes = plt.subplots(2, 5, figsize=(24, 10))
    for col, key in enumerate(order):
        for row, (um, dist_label) in enumerate([(um_a, 'Hard-A'), (um_b, 'Hard-B')]):
            ax = axes[row, col]
            u = um.get(key)
            if u is not None:
                plot_um(u, ax=ax, title=f'{dist_label} {key} (n={u["n_trials"]})',
                        vmin=-0.35, vmax=0.35, colorbar=False)
            else:
                ax.text(0.5, 0.5, f'{dist_label} {key}\nNo data', ha='center',
                        va='center', transform=ax.transAxes, color='grey')
                ax.set_axis_off()
    fig.suptitle(f'{tag} — Hard Update Matrices', fontsize=14)
    fig.tight_layout()
    return fig

In [ ]:
def plot_phase_single(psyc, um, title):
    keys = [k for k in psyc if psyc[k] is not None and psyc[k].get('success')]
    if not keys:
        return None
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    key = keys[0]
    plot_psychometric(psyc[key], ax=axes[0], color=CC.get(key, 'grey'),
                      label=f'{key} (n={psyc[key]["n_trials"]})')
    axes[0].legend(fontsize=8); axes[0].set_title('Psychometric')
    u = um.get(key)
    if u is not None:
        plot_um(u, ax=axes[1], title=f'{key} (n={u["n_trials"]})',
                vmin=-0.35, vmax=0.35, colorbar=False)
    fig.suptitle(title, fontsize=14); fig.tight_layout()
    return fig


def plot_nonopto_hard(psyc_a, psyc_b, um_a, um_b, tag):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    pa, pb = psyc_a.get('regular'), psyc_b.get('regular')
    if pa and pa.get('success'):
        plot_psychometric(pa, ax=axes[0], color=HARD_A_COLOUR, label=f'Hard-A (n={pa["n_trials"]})')
    if pb and pb.get('success'):
        plot_psychometric(pb, ax=axes[0], color=HARD_B_COLOUR, label=f'Hard-B (n={pb["n_trials"]})')
    axes[0].set_title('A vs B: Psychometric', fontsize=11); axes[0].legend(fontsize=8)
    for i, (um_d, dist) in enumerate([(um_a, 'Hard-A'), (um_b, 'Hard-B')]):
        u = um_d.get('regular')
        if u is not None:
            plot_um(u, ax=axes[i+1], title=f'{dist} (n={u["n_trials"]})',
                    vmin=-0.35, vmax=0.35, colorbar=False)
        else:
            axes[i+1].text(0.5, 0.5, f'{dist}\nNo data', ha='center',
                           va='center', transform=axes[i+1].transAxes, color='grey')
    fig.suptitle(f'{tag} — Hard-A vs Hard-B', fontsize=14); fig.tight_layout()
    return fig

In [ ]:
PHASE_LABELS = {'uniform': 'Uniform', 'hard_a': 'Hard-A', 'hard_b': 'Hard-B'}
PHASE_GAP = 1.5
BOX_WIDTH = 0.6
DOT_SIZE = 25
DOT_ALPHA = 0.7


def plot_strip(stats_df, stats_to_plot, title, phase_order=PHASE_ORDER):
    """
    Grouped box + dots. Returns list[Figure], one page per stat.

    Parameters
    ----------
    stats_df : DataFrame with 'phase', 'condition', and stat columns.
    stats_to_plot : list of (column_name, display_label).
    title : str
    """
    rng = np.random.default_rng(42)
    figures = []

    seen = []
    for _, row in stats_df.iterrows():
        c = row['condition']
        if c not in seen and c in CC:
            seen.append(c)
    legend_handles = [Line2D([0],[0], marker='o', color='w',
                             markerfacecolor=CC[c], markersize=8, label=c)
                      for c in seen]

    for col, ylabel in stats_to_plot:
        fig, ax = plt.subplots(figsize=(10, 5))

        if col not in stats_df.columns or stats_df[col].dropna().empty:
            ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                    transform=ax.transAxes, color='grey', fontsize=14)
            ax.set_ylabel(ylabel)
            fig.suptitle(f'{title} — {ylabel}', fontsize=14)
            figures.append(fig)
            continue

        x_pos = 0
        tick_positions, tick_labels = [], []
        group_spans = []

        for phase in phase_order:
            phase_df = stats_df[stats_df['phase'] == phase]
            if phase_df.empty:
                continue
            conditions = [c for c in phase_df['condition'].unique() if c in CC]
            if not conditions:
                continue

            group_start = x_pos
            for cond in conditions:
                vals = phase_df.loc[phase_df['condition'] == cond, col].dropna().values
                if len(vals) == 0:
                    x_pos += 1
                    continue

                colour = CC[cond]
                bp = ax.boxplot(
                    [vals], positions=[x_pos], widths=BOX_WIDTH,
                    patch_artist=True, showfliers=False,
                    medianprops=dict(color='k', linewidth=1.5),
                    whiskerprops=dict(color=colour),
                    capprops=dict(color=colour),
                )
                bp['boxes'][0].set_facecolor(colour)
                bp['boxes'][0].set_alpha(0.35)
                bp['boxes'][0].set_edgecolor(colour)

                jitter = rng.normal(0, 0.07, len(vals))
                ax.scatter(x_pos + jitter, vals, s=DOT_SIZE, c=colour,
                           edgecolors='white', linewidths=0.4,
                           alpha=DOT_ALPHA, zorder=3)

                tick_positions.append(x_pos)
                tick_labels.append(cond)
                x_pos += 1

            group_end = x_pos - 1
            group_spans.append((group_start, group_end, PHASE_LABELS.get(phase, phase)))
            x_pos += PHASE_GAP

        ax.set_xticks(tick_positions)
        ax.set_xticklabels(tick_labels, fontsize=9, rotation=35, ha='right')
        for start, end, label in group_spans:
            ax.text((start + end) / 2, ax.get_ylim()[1], label,
                    ha='center', va='bottom', fontsize=12,
                    fontweight='bold', color='#444444')
        ax.set_ylabel(ylabel, fontsize=12)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.legend(handles=legend_handles, fontsize=8, loc='upper right')
        fig.suptitle(f'{title} — {ylabel}', fontsize=14)
        fig.tight_layout(rect=[0, 0, 1, 0.93])
        figures.append(fig)

    return figures


DEFAULT_STRIP_STATS = [
    ('n_trials', 'N Trials'),
    ('accuracy', 'Accuracy'),
    ('win_stay', 'Win-Stay'),
    ('lose_shift', 'Lose-Shift'),
]

## Assemble & Save

In [ ]:
def generate_animal_report(animal, aid, strip_stats=DEFAULT_STRIP_STATS):
    genotype = animal.genotype
    tag = f'{aid} ({genotype.upper()})'
    opto = is_opto_cohort(animal)
    cohort = 'opto' if opto else 'non-opto'
    figures = []

    figures.append(plot_session_timeline(animal, aid))
    figures.append(plot_trajectory_page(animal, tag))

    # ── Full data ─────────────────────────────────────────────
    uni_clean, uni_psyc, uni_um = compute_phase(animal, 'uniform', cohort)
    if opto:
        figures.append(plot_uniform_psychometric(uni_psyc, tag))
        figures.append(plot_uniform_um(uni_um, tag))
    else:
        fig = plot_phase_single(uni_psyc, uni_um, f'{tag} — Uniform')
        if fig is not None:
            figures.append(fig)

    ha_clean, ha_psyc, ha_um = compute_phase(animal, 'hard_a', cohort)
    hb_clean, hb_psyc, hb_um = compute_phase(animal, 'hard_b', cohort)
    has_hard = (any(v is not None for v in ha_psyc.values()) or
                any(v is not None for v in hb_psyc.values()))
    if has_hard:
        if opto:
            figures.append(plot_hard_psychometric(ha_psyc, hb_psyc, tag))
            figures.append(plot_hard_um(ha_um, hb_um, tag))
        else:
            fig = plot_nonopto_hard(ha_psyc, hb_psyc, ha_um, hb_um, tag)
            if fig is not None:
                figures.append(fig)

    # ── Downsampled (matched n) ───────────────────────────────
    if opto:
        ds_uni_psyc, ds_uni_um = downsample_phase(uni_clean, n_repeats=100)
        figures.append(plot_uniform_psychometric(ds_uni_psyc, f'{tag} — Uniform (matched n)'))
        figures.append(plot_uniform_um(ds_uni_um, f'{tag} — Uniform UMs (matched n)'))

        if has_hard:
            ds_ha_psyc, ds_ha_um = downsample_phase(ha_clean, n_repeats=100)
            ds_hb_psyc, ds_hb_um = downsample_phase(hb_clean, n_repeats=100)
            figures.append(plot_hard_psychometric(
                ds_ha_psyc, ds_hb_psyc, f'{tag} — Hard (matched n)'))
            figures.append(plot_hard_um(
                ds_ha_um, ds_hb_um, f'{tag} — Hard UMs (matched n)'))

    # ── Strip ─────────────────────────────────────────────────
    stat_cols = [col for col, _ in strip_stats]
    strip_df = build_strip_df(animal, stats_of_interest=stat_cols, cohort=cohort)
    figures.extend(plot_strip(strip_df, strip_stats, tag))

    return figures, strip_df


def save_pdf(figures, path):
    with PdfPages(path) as pdf:
        for fig in figures:
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)

## Interactive: Single Animal

In [ ]:
animal_id = 'SS21'  # ← change this
animal = experiment.get_animal(animal_id)
tag = f'{animal_id} ({animal.genotype.upper()})'
opto = is_opto_cohort(animal)
print(f'{animal_id} | {animal.genotype} | {"opto" if opto else "non-opto"}')

In [ ]:
plot_session_timeline(animal, animal_id); plt.show()
plot_trajectory_page(animal, tag); plt.show()

In [ ]:
# filter_phase: select sessions + filter trials in one call (the new primitive).
# Each report panel is one filter_phase spec; here, the uniform opto-off control trials.
opto_off = filter_phase(animal, dist='uniform', session_type='opto', trial_type='opto_off')
n = sum(int(s.trials.valid_mask.sum()) for s in opto_off)
print(f'uniform opto-off: {len(opto_off)} sessions, {n} trials')

In [ ]:
_, uni_psyc, uni_um = compute_phase(animal, 'uniform')
if opto:
    fig = plot_uniform_psychometric(uni_psyc, tag); plt.show()
    plot_uniform_um(uni_um, tag); plt.show()

In [ ]:
_, ha_psyc, ha_um = compute_phase(animal, 'hard_a')
_, hb_psyc, hb_um = compute_phase(animal, 'hard_b')
if opto:
    plot_hard_psychometric(ha_psyc, hb_psyc, tag); plt.show()
    plot_hard_um(ha_um, hb_um, tag); plt.show()

### Interactive: Downsampled

In [ ]:
# Downsampled (matched n) — uniform. downsample_phase wraps the matched-n draw
# (calculate_min_n + compute_ds_x per condition); here we just read off the result.
uni_clean, _, _ = compute_phase(animal, 'uniform')
clean_sessions = [s for s in uni_clean.values() if s]

print(f"matched n: {calculate_min_n(clean_sessions, unit='trials')} trials, "
      f"{calculate_min_n(clean_sessions, unit='pairs')} pairs")
ds_uni_psyc, ds_uni_um = downsample_phase(uni_clean, n_repeats=50)


for label in uni_clean:
    p = ds_uni_psyc.get(label)
    print(f"  {label}: " + ("None" if p is None else
                            f"psyc n={p['n_trials']}, success={p['success']}"))

In [ ]:
# Plot downsampled if available
if any(v is not None for v in ds_uni_psyc.values()):
    plot_uniform_psychometric(ds_uni_psyc, f'{tag} — Uniform (matched n)'); plt.show()
else:
    print("All downsampled psychometrics are None — check diagnostics above")

if any(v is not None for v in ds_uni_um.values()):
    plot_uniform_um(ds_uni_um, f'{tag} — Uniform UMs (matched n)'); plt.show()

In [ ]:
# Downsampled (matched n) — uniform. downsample_phase wraps the matched-n draw
# (calculate_min_n + compute_ds_x per condition); here we just read off the result.
uni_clean, _, _ = compute_phase(animal, 'hard_a')
clean_sessions = [s for s in uni_clean.values() if s]

print(f"matched n: {calculate_min_n(clean_sessions, unit='trials')} trials, "
      f"{calculate_min_n(clean_sessions, unit='pairs')} pairs")
ds_uni_psyc, ds_uni_um = downsample_phase(uni_clean, n_repeats=50)


for label in uni_clean:
    p = ds_uni_psyc.get(label)
    print(f"  {label}: " + ("None" if p is None else
                            f"psyc n={p['n_trials']}, success={p['success']}"))
    
_, ha_psyc, ha_um = compute_phase(animal, 'hard_a')
_, hb_psyc, hb_um = compute_phase(animal, 'hard_b')

if opto:
    plot_hard_psychometric(ha_psyc, hb_psyc, tag); plt.show()
    plot_hard_um(ha_um, hb_um, tag); plt.show()

In [ ]:
# Strip — one page per stat
strip_df = build_strip_df(animal)
for fig in plot_strip(strip_df, DEFAULT_STRIP_STATS, tag):
    plt.show()

In [ ]:
# Custom stats
custom_stats = [('accuracy', 'Accuracy'), ('win_stay', 'Win-Stay'),
                ('lose_shift', 'Lose-Shift'), ('side_bias', 'Side Bias')]
strip_df2 = build_strip_df(animal, stats_of_interest=[c for c, _ in custom_stats])
for fig in plot_strip(strip_df2, custom_stats, tag):
    plt.show()

## Batch: All Animals → PDF

In [ ]:
# OUTPUT_DIR = Path('outputs/animal_reports')
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# for aid, animal in sorted(experiment.animals.items()):
#     cohort = 'opto' if is_opto_cohort(animal) else 'non-opto'
#     print(f'{aid} | {animal.genotype} | {cohort} ...', end=' ')
#     figures, strip_df = generate_animal_report(animal, aid)
#     pdf_path = OUTPUT_DIR / f'{aid}_{animal.genotype}_{cohort}.pdf'
#     save_pdf(figures, pdf_path)
#     print(f'{len(figures)} pages → {pdf_path.name}')

# print(f'\nDone. Reports in {OUTPUT_DIR}/')